In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
dataset = pd.read_csv("insurance_pre.csv")

In [3]:
dataset

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520
...,...,...,...,...,...,...
1333,50,male,30.970,3,no,10600.54830
1334,18,female,31.920,0,no,2205.98080
1335,18,female,36.850,0,no,1629.83350
1336,21,female,25.800,0,no,2007.94500


In [4]:
dataset = pd.get_dummies(dataset, dtype=int, drop_first=True)

In [5]:
dataset

,age,bmi,children,charges,sex_male,smoker_yes
0,19,27.900,0,16884.92400,0,1
1,18,33.770,1,1725.55230,1,0
2,28,33.000,3,4449.46200,1,0
3,33,22.705,0,21984.47061,1,0
4,32,28.880,0,3866.85520,1,0
...,...,...,...,...,...,...
1333,50,30.970,3,10600.54830,1,0
1334,18,31.920,0,2205.98080,0,0
1335,18,36.850,0,1629.83350,0,0
1336,21,25.800,0,2007.94500,0,0


In [6]:
dataset.columns

Index(['age', 'bmi', 'children', 'charges', 'sex_male', 'smoker_yes'], dtype='str')

In [7]:
independent = dataset[['age', 'bmi', 'children','sex_male', 'smoker_yes']]

In [8]:
independent

,age,bmi,children,sex_male,smoker_yes
0,19,27.900,0,0,1
1,18,33.770,1,1,0
2,28,33.000,3,1,0
3,33,22.705,0,1,0
4,32,28.880,0,1,0
...,...,...,...,...,...
1333,50,30.970,3,1,0
1334,18,31.920,0,0,0
1335,18,36.850,0,0,0
1336,21,25.800,0,0,0


In [9]:
dependent = dataset[['charges']]

In [10]:
dependent

,charges
0,16884.92400
1,1725.55230
2,4449.46200
3,21984.47061
4,3866.85520
...,...
1333,10600.54830
1334,2205.98080
1335,1629.83350
1336,2007.94500


In [11]:
from sklearn.model_selection import train_test_split

In [12]:
x_test, x_train, y_test, y_train = train_test_split(independent, dependent, test_size=0.30, random_state=0)

In [13]:
x_test

,age,bmi,children,sex_male,smoker_yes
1163,18,28.215,0,0,0
196,39,32.800,0,0,0
438,52,46.750,5,0,0
183,44,26.410,0,0,0
1298,33,27.455,2,1,0
...,...,...,...,...,...
763,27,26.030,0,1,0
835,42,35.970,2,1,0
1216,40,25.080,0,1,0
559,19,35.530,0,1,0


Output prediction value is 80 so improving that adding standardisation

In [14]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [22]:
x_train

array([[ 0.94974655, -0.04448183, -0.10502969,  0.91412058, -0.51395617],
       [ 0.5955739 , -0.18856306, -0.10502969, -1.09394758, -0.51395617],
       [ 0.66640843,  1.75479764,  0.72285141,  0.91412058,  1.94569121],
       ...,
       [ 1.16225013,  1.35814027, -0.93291079,  0.91412058, -0.51395617],
       [ 1.79976089,  1.5109011 , -0.10502969,  0.91412058, -0.51395617],
       [ 1.65809183, -0.94715942, -0.93291079, -1.09394758, -0.51395617]],
      shape=(402, 5))

In [45]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
param_grid = {
    'criterion':['squared_error','absolute_error','poisson'],
    'max_features':[None,'sqrt','log2'],
    'splitter':['best','random']
}
grid = GridSearchCV(
    estimator=DecisionTreeRegressor(),
    param_grid=param_grid,
    refit=True,
    verbose=3,
    n_jobs=-1
)
grid.fit(x_train, y_train)

Fitting 5 folds for each of 18 candidates, totalling 90 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",DecisionTreeRegressor()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'criterion': ['squared_error', 'absolute_error', ...], 'max_features': [None, 'sqrt', ...], 'splitter': ['best', 'random']}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",3
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multicl

In [46]:
re=grid.cv_results_
print("TheR_score valueforbestparameter{}:".format(grid.best_params_))

TheR_score valueforbestparameter{'criterion': 'poisson', 'max_features': None, 'splitter': 'best'}:


This shows how many support vector it is being used

In [47]:
 table=pd.DataFrame.from_dict(re)

In [48]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.015449,0.005969,0.000945,0.000219,squared_error,NaN,best,"{'criterion': 'squared_error', 'max_features':...",0.847953,0.765431,0.784381,0.702321,0.788750,0.777767,0.046765,3
1,0.017437,0.014562,0.001690,0.000759,squared_error,NaN,random,"{'criterion': 'squared_error', 'max_features':...",0.778692,0.734751,0.639253,0.818329,0.894807,0.773166,0.085177,4
2,0.038142,0.018008,0.002260,0.000627,squared_error,sqrt,best,"{'criterion': 'squared_error', 'max_features':...",0.620953,0.736102,0.666975,0.571551,0.561601,0.631437,0.064484,17
3,0.030673,0.023130,0.002041,0.000626,squared_error,sqrt,random,"{'criterion': 'squared_error', 'max_features':...",0.737396,0.722722,0.657677,0.585496,0.538864,0.648431,0.076792,16
4,0.020855,0.022687,0.002037,0.000618,squared_error,log2,best,"{'criterion': 'squared_error', 'max_features':...",0.767641,0.835742,0.613892,0.569854,0.828662,0.723158,0.110658,10
5,0.002486,0.000644,0.001826,0.000619,squared_error,log2,random,"{'criterion': 'squared_error', 'max_features':...",0.594736,0.799149,0.597386,0.417392,0.724280,0.626589,0.130347,18
6,0.006581,0.000674,0.002307,0.000464,absolute_error,NaN,best,"{'criterion': 'absolute_error', 'max_features'...",0.850229,0.823299,0.710188,0.717110,0.821888,0.784543,0.058800,2
7,0.006511,0.000999,0.002122,0.000443,absolute_error,NaN,random,"{'criterion': 'absolute_error', 'max_features'...",0.867353,0.863158,0.622920,0.579823,0.767909,0.740233,0.119607,8
8,0.004734,0.001092,0.002284,0.000406,absolute_error,sqrt,best,"{'criterion': 'absolute_error', 'max_features'...",0.732086,0.416379,0.622485,0.733927,0.747416,0.650459,0.125385,15
9,0.003865,0.000714,0.001863,0.000495,absolute_error,sqrt,random,"{'criterion': 'absolute_error', 'max_features'...",0.615787,0.707127,0.653213,0.734149,0.794048,0.700865,0.062192,12


In [51]:
age_input=float(input("Age:"))
bmi_input=float(input("BMI:"))
children_input=float(input("Children:"))
sex_male_input=int(input("Sex Male 0 or 1:"))
smoker_yes_input=int(input("Smoker Yes 0 or 1:"))

Age: 23
BMI: 44
Children: 55
Sex Male 0 or 1: 1
Smoker Yes 0 or 1: 1


In [52]:
Future_Prediction=grid.predict([[age_input,bmi_input,children_input,sex_male_input,smoker_yes_input]])
#change the paramter,play with it.
print("Future_Prediction={}".format(Future_Prediction))

Future_Prediction=[48517.56315]
